In [1]:
#Amo demo
#GV 19.9.2025 + FM 22/10/2025 + FM 05.11.2025
import sys
sys.path.append('amos') 
# Import all ML orchestration functions
from amos.ml_orchestration import (
    ml_exp_with_timing,  # Main experiment function with timing preservation
    extract_timing_structure,
    save_midi_with_exact_timing_structure,
    KerasClassifierWrapper,
    build_lstm_classifier,
    build_transformer_classifier,
    clf_predict,
    defineXy,           # Data preparation function
    split_and_encode    # Train/test split with encoding
)
# Import MIDI processing functions  
from amos.midi2df2midi import midi_to_dataframe, save_midi_from_df
from amos.mappings import fill_quaterna_columns, learn_quaterna_mapping
# Import amo funtion
from amos.ml_orchestration import amo_with_doublings_multiclass

import pandas as pd
import numpy as np
from collections import defaultdict

/home/francesco/anaconda3/envs/auto-orch/lib/python3.12/site-packages/xgboost/core.py:377: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc >= 2.28) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain features such as GPU algorithms or federated learning are not available. To use these features, please upgrade to a recent Linux distro with glibc 2.28+, and install the 'manylinux_2_28' variant.
  warnings.warn(
2025-12-09 14:32:41.626093: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1765287161.659558   98034 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been

In [2]:
# amo_with_doublings
# Args:
  #      filein (str): Path to the source MIDI file to learn orchestration style from.
  #      fileout (str): Path to the target MIDI file to be orchestrated.
  #      ytarget (str, optional): The target variable for the model ('track-channel' or 'program').
  #                               Defaults to "track-channel".
  #      model (str, optional): The name of the machine learning model to use for orchestration.
  #     ("XGBoost","RandomForest", "DecisionTree", "NearestNeighbors", "MLP3", "NaiveBayes", "MLP1", "AdaBoost", "LSTMClassifier", "TransformerClassifier")
  #                             Defaults to "XGBoost".

In [3]:
def transpose(note, inverse=False, n_semitones=12):
    # Example: transpose pitch
    if inverse:
        n_semitones = - n_semitones
    new_note = note.copy()
    new_note['pitch'] = note['pitch'] + n_semitones
    return [ new_note ]

In [4]:
transformations = [
    (transpose, {'n_semitones': 12}),
    (transpose, {'n_semitones': 24}),
    (transpose, {'n_semitones': 46}),
    (transpose, {'n_semitones': -12}),
    (transpose, {'n_semitones': -24}),
    (transpose, {'n_semitones': -46}),
    #(split_duration, {'smallest_unit': 0.25}),
]

In [5]:
model="XGBoost5"
#model="AdaBoost"

In [6]:
filein='midis/symphony_7_1_orch.mid'
fileout='midis/Autumn.mid'

# Multiclass, with transformations
amo_with_doublings_multiclass(filein,fileout,model=model, tol=0.2, transformations=transformations)

Learning orchestration style from: midis/symphony_7_1_orch.mid

========= PREPROCESSING =========
Mapping: [[1 'Flutes' 0 73]
 [2 'Oboes' 1 68]
 [3 'Clarinets in A' 2 71]
 [4 'Bassoons' 3 70]
 [5 'Horns in A' 4 60]
 [6 'Trumpets in D' 5 56]
 [7 'Timpani in A E' 6 47]
 [8 '1st Violins' 7 48]
 [9 '2nd Violins' 8 48]
 [10 'Violas' 10 48]
 [11 'Cellos/Basses' 11 45]
 [11 'Cellos/Basses' 11 48]
 [12 'Cellos/Basses' 12 45]
 [12 'Cellos/Basses' 12 48]]

Building reduced dataset with transformations
{'transformed': 9867, 'direct': 1321, 'none': 10626}
Original size: (21814, 15)
Dropping [1690, 3480, 3481, 5293, 7205, 7206, 8352, 8353, 9008, 9655, 9656, 11456, 14676, 14677, 14678, 18036, 19925, 5295, 7207, 8354, 9009, 9657, 9658, 11458, 11459, 14680, 14681, 14682, 18037, 19926, 5297, 8356, 9010, 9659, 9660, 11461, 11462, 14684, 14685, 14686, 18038, 19927, 7211, 8357, 8358, 9011, 9661, 9662, 11465, 11466, 14688, 18039, 19928, 5299, 7213, 14690, 19929, 5300, 5301, 19930, 1710, 3498, 3500, 11469, 

{'time': 0.454770565032959,
 'accuracy (test)': np.float64(0.3829787234042553),
 'accuracy (train)': np.float64(0.39908190032060625),
 'precision (test)': 0.7993934142114385,
 'precision (train)': 0.8263623457464235,
 'recall (test)': 0.4224868330661782,
 'recall (train)': 0.43371353241244914,
 'f1 (test)': 0.5528089887640449,
 'f1 (train)': 0.5688618252894302,
 "time (transpose_{'n_semitones': 12})": 0.7109501361846924,
 "accuracy (test) (transpose_{'n_semitones': 12})": 0.841486359360301,
 "accuracy (train) (transpose_{'n_semitones': 12})": 0.9221176470588235,
 "precision (test) (transpose_{'n_semitones': 12})": 0.8382232437773556,
 "precision (train) (transpose_{'n_semitones': 12})": 0.9219946116498411,
 "recall (test) (transpose_{'n_semitones': 12})": 0.841486359360301,
 "recall (train) (transpose_{'n_semitones': 12})": 0.9221176470588235,
 "f1 (test) (transpose_{'n_semitones': 12})": 0.8379887379048854,
 "f1 (train) (transpose_{'n_semitones': 12})": 0.9215419973015311,
 "time (tra

In [7]:
# Multiclass, no transformations
amo_with_doublings_multiclass(filein,fileout,model=model, tol=0.2, transformations=None)

Learning orchestration style from: midis/symphony_7_1_orch.mid

========= PREPROCESSING =========
Mapping: [[1 'Flutes' 0 73]
 [2 'Oboes' 1 68]
 [3 'Clarinets in A' 2 71]
 [4 'Bassoons' 3 70]
 [5 'Horns in A' 4 60]
 [6 'Trumpets in D' 5 56]
 [7 'Timpani in A E' 6 47]
 [8 '1st Violins' 7 48]
 [9 '2nd Violins' 8 48]
 [10 'Violas' 10 48]
 [11 'Cellos/Basses' 11 45]
 [11 'Cellos/Basses' 11 48]
 [12 'Cellos/Basses' 12 45]
 [12 'Cellos/Basses' 12 48]]

Creating multi-hot encoding for notes with instrumental doubling
Number of classes: 12
Classes: [np.str_('10_10'), np.str_('11_11'), np.str_('12_12'), np.str_('1_0'), np.str_('2_1'), np.str_('3_2'), np.str_('4_3'), np.str_('5_4'), np.str_('6_5'), np.str_('7_6'), np.str_('8_7'), np.str_('9_8')]
Number of events in midis/symphony_7_1_orch.mid : 17155
Last onset at 1419.0
[[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 1 0]
 ...
 [1 0 0 ... 0 0 0]
 [0 0 1 ... 0 0 0]
 [0 0 0 ... 0 1 0]]
[1 1 3 ... 1 1 1]
6

Processing target file: midis/Autumn

{'time': 0.42830371856689453,
 'accuracy (test)': np.float64(0.3829787234042553),
 'accuracy (train)': np.float64(0.39908190032060625),
 'precision (test)': 0.7993934142114385,
 'precision (train)': 0.8263623457464235,
 'recall (test)': 0.4224868330661782,
 'recall (train)': 0.43371353241244914,
 'f1 (test)': 0.5528089887640449,
 'f1 (train)': 0.5688618252894302}

In [8]:
# Single class, with transformations
amo_with_doublings_multiclass(filein,fileout,model=model, tol=0.2, transformations=transformations, multiclass=False)

Learning orchestration style from: midis/symphony_7_1_orch.mid

========= PREPROCESSING =========
Mapping: [[1 'Flutes' 0 73]
 [2 'Oboes' 1 68]
 [3 'Clarinets in A' 2 71]
 [4 'Bassoons' 3 70]
 [5 'Horns in A' 4 60]
 [6 'Trumpets in D' 5 56]
 [7 'Timpani in A E' 6 47]
 [8 '1st Violins' 7 48]
 [9 '2nd Violins' 8 48]
 [10 'Violas' 10 48]
 [11 'Cellos/Basses' 11 45]
 [11 'Cellos/Basses' 11 48]
 [12 'Cellos/Basses' 12 45]
 [12 'Cellos/Basses' 12 48]]

Building reduced dataset with transformations
{'transformed': 9867, 'direct': 1321, 'none': 10626}
Original size: (21814, 15)
Dropping [1690, 3480, 3481, 5293, 7205, 7206, 8352, 8353, 9008, 9655, 9656, 11456, 14676, 14677, 14678, 18036, 19925, 5295, 7207, 8354, 9009, 9657, 9658, 11458, 11459, 14680, 14681, 14682, 18037, 19926, 5297, 8356, 9010, 9659, 9660, 11461, 11462, 14684, 14685, 14686, 18038, 19927, 7211, 8357, 8358, 9011, 9661, 9662, 11465, 11466, 14688, 18039, 19928, 5299, 7213, 14690, 19929, 5300, 5301, 19930, 1710, 3498, 3500, 11469, 

{'time': 0.12297868728637695,
 'accuracy (test)': 0.5487050194820078,
 'accuracy (train)': 0.5748094665062174,
 'precision (test)': 0.5574098913674037,
 'precision (train)': 0.5748094665062174,
 'recall (test)': 0.5487050194820078,
 'recall (train)': 0.5748094665062174,
 'f1 (test)': 0.5406901525478699,
 'f1 (train)': 0.5748094665062174,
 "time (transpose_{'n_semitones': 12})": 0.7371082305908203,
 "accuracy (test) (transpose_{'n_semitones': 12})": 0.841486359360301,
 "accuracy (train) (transpose_{'n_semitones': 12})": 0.9221176470588235,
 "precision (test) (transpose_{'n_semitones': 12})": 0.8382232437773556,
 "precision (train) (transpose_{'n_semitones': 12})": 0.9219946116498411,
 "recall (test) (transpose_{'n_semitones': 12})": 0.841486359360301,
 "recall (train) (transpose_{'n_semitones': 12})": 0.9221176470588235,
 "f1 (test) (transpose_{'n_semitones': 12})": 0.8379887379048854,
 "f1 (train) (transpose_{'n_semitones': 12})": 0.9215419973015311,
 "time (transpose_{'n_semitones': 2

In [9]:
# Single class, no transformations
amo_with_doublings_multiclass(filein,fileout,model=model, tol=0.2, transformations=None, multiclass=False)

Learning orchestration style from: midis/symphony_7_1_orch.mid

========= PREPROCESSING =========
Mapping: [[1 'Flutes' 0 73]
 [2 'Oboes' 1 68]
 [3 'Clarinets in A' 2 71]
 [4 'Bassoons' 3 70]
 [5 'Horns in A' 4 60]
 [6 'Trumpets in D' 5 56]
 [7 'Timpani in A E' 6 47]
 [8 '1st Violins' 7 48]
 [9 '2nd Violins' 8 48]
 [10 'Violas' 10 48]
 [11 'Cellos/Basses' 11 45]
 [11 'Cellos/Basses' 11 48]
 [12 'Cellos/Basses' 12 45]
 [12 'Cellos/Basses' 12 48]]

Define covariates and target variable. Target variable encoding
Labels ['10_10' '11_11' '12_12' '1_0' '2_1' '3_2' '4_3' '5_4' '6_5' '7_6' '8_7'
 '9_8']
Number of events in midis/symphony_7_1_orch.mid : 21814
Last onset at 1419.0

Processing target file: midis/Autumn.mid
Number of events in midis/Autumn.mid : 4915
Last onset at 841.75
Original ticks_per_beat: 256
y_train, test size: 0.2 , labels: ['10_10' '11_11' '12_12' '1_0' '2_1' '3_2' '4_3' '5_4' '6_5' '7_6' '8_7'
 '9_8']
Number of unknown classes in test set: 0 of 0.2
y_train, test size: 0

{'time': 0.17329621315002441,
 'accuracy (test)': 0.5487050194820078,
 'accuracy (train)': 0.5748094665062174,
 'precision (test)': 0.5574098913674037,
 'precision (train)': 0.5748094665062174,
 'recall (test)': 0.5487050194820078,
 'recall (train)': 0.5748094665062174,
 'f1 (test)': 0.5406901525478699,
 'f1 (train)': 0.5748094665062174}